# DeepDTA on Davis — Unified Colab Notebook

This is the corrected, restartable version of the uploaded browser-inference notebook.

**Workflow:** Davis data → original fold split → character encoding → two-branch CNN → five-fold validation → final training → held-out test evaluation → model/config export → TensorFlow.js browser app → Hugging Face Static Space.

**Persistence:** all project outputs are stored in Google Drive. The original DeepDTA repository is read-only.

**Important:** the notebook's configured input lengths (85 and 1,200) intentionally follow the supplied browser notebook. Some Davis sequences are longer and will be truncated; the preprocessing section reports how many. Do not compare results directly with a paper benchmark unless preprocessing and evaluation protocol match.

Run cells in order. After a Colab reconnect, mount Drive and rerun setup/data cells; completed checkpoints and exported artifacts remain in Drive.


## 1. Setup and permanent Google Drive storage


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import pickle
import shutil
import subprocess
import sys
import time
import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_DIR = "/content/drive/MyDrive/DeepDTA_Unified_Fresh"
# Reuse the already-downloaded original repository as read-only source data.
REPO_DIR = "/content/drive/MyDrive/DeepDTA_Unified/original_deepdta"
DAVIS_DIR = os.path.join(REPO_DIR, "data", "davis")

MODEL_DIR = os.path.join(PROJECT_DIR, "artifacts", "model")
RESULTS_DIR = os.path.join(PROJECT_DIR, "artifacts", "results")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints", "davis")
DATASET_DIR = os.path.join(PROJECT_DIR, "datasets")
WEB_DIR = os.path.join(PROJECT_DIR, "web")

for folder in [MODEL_DIR, RESULTS_DIR, CHECKPOINT_DIR, DATASET_DIR, WEB_DIR]:
    os.makedirs(folder, exist_ok=True)

assert os.path.isdir(DAVIS_DIR), (
    "Davis dataset not found. Confirm the original_deepdta/data/davis folder "
    "exists in DeepDTA_Unified on Google Drive."
)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Project:", PROJECT_DIR)
print("Davis data:", DAVIS_DIR)


## 2. Experiment settings
These values follow the uploaded browser notebook. The sequence-length truncation report appears in Section 7.


In [ ]:
# Input lengths from the supplied browser notebook
MAX_SMILES_LENGTH = 85
MAX_PROTEIN_LENGTH = 1200

# Original DeepDTA categorical CNN design
EMBEDDING_DIM = 128
NUM_FILTERS = 32                  # convolution widths: 32, 64, 96
SMILES_KERNEL = 4
PROTEIN_KERNEL = 8
DENSE_UNITS = (1024, 1024, 512)
DROPOUT_RATE = 0.1

# Training
LEARNING_RATE = 0.001
BATCH_SIZE = 256
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 7

# Davis pKd interpretation used for AUPR / interface
PKD_THRESHOLD = 7.0

CONFIG = {
    "dataset": "Davis",
    "target": "pKd = -log10(Kd in molar)",
    "max_smiles_length": MAX_SMILES_LENGTH,
    "max_protein_length": MAX_PROTEIN_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "num_filters": [NUM_FILTERS, NUM_FILTERS * 2, NUM_FILTERS * 3],
    "smiles_kernel": SMILES_KERNEL,
    "protein_kernel": PROTEIN_KERNEL,
    "dense_units": list(DENSE_UNITS),
    "dropout_rate": DROPOUT_RATE,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "pkd_threshold": PKD_THRESHOLD,
    "inference": "TensorFlow.js in the browser"
}

print(json.dumps(CONFIG, indent=2))


## 3. Imports and original Davis data


In [ ]:
from collections import OrderedDict
from sklearn.metrics import mean_squared_error, average_precision_score

# We use the already-preserved original repository in Drive.
# No second clone and no edits to the parent repository are made.
required_files = [
    "ligands_can.txt",
    "proteins.txt",
    "Y",
    os.path.join("folds", "train_fold_setting1.txt"),
    os.path.join("folds", "test_fold_setting1.txt")
]
missing = [name for name in required_files if not os.path.isfile(os.path.join(DAVIS_DIR, name))]
assert not missing, f"Missing Davis files: {missing}"

print("Original Davis files verified.")
print("Original source remains read-only.")


## 4. Load the Davis dataset
Drug inputs are SMILES strings, protein inputs are amino-acid sequences, and `Y` contains Kd values in nM.


In [ ]:
with open(os.path.join(DAVIS_DIR, "ligands_can.txt"), encoding="utf-8") as f:
    ligands = json.load(f, object_pairs_hook=OrderedDict)

with open(os.path.join(DAVIS_DIR, "proteins.txt"), encoding="utf-8") as f:
    proteins = json.load(f, object_pairs_hook=OrderedDict)

with open(os.path.join(DAVIS_DIR, "Y"), "rb") as f:
    Y_nM = np.asarray(pickle.load(f, encoding="latin1"), dtype=np.float64)

drugs = list(ligands.values())
targets = list(proteins.values())

assert Y_nM.shape == (len(drugs), len(targets))
print("Drugs:", len(drugs))
print("Proteins:", len(targets))
print("Affinity matrix:", Y_nM.shape)
print("Raw Kd range (nM):", np.nanmin(Y_nM), "to", np.nanmax(Y_nM))
print("Missing raw measurements:", int(np.isnan(Y_nM).sum()))


## 5. Convert Kd to pKd
For Kd recorded in nM, `pKd = 9 - log10(Kd_nM)`. NaN entries remain missing.


In [ ]:
affinity = 9.0 - np.log10(Y_nM)
drug_idx, target_idx = np.where(np.isfinite(affinity))

print("Known drug-target pairs:", len(drug_idx))
print("pKd range:", float(np.nanmin(affinity)), "to", float(np.nanmax(affinity)))
print("pKd values at 5.0:", int(np.isclose(affinity[np.isfinite(affinity)], 5.0).sum()))


## 6. Official training and held-out test folds
The five official training folds together form the model-development set. The official test fold is kept untouched until final evaluation.


In [ ]:
with open(os.path.join(DAVIS_DIR, "folds", "train_fold_setting1.txt"), encoding="utf-8") as f:
    train_folds = json.load(f)

with open(os.path.join(DAVIS_DIR, "folds", "test_fold_setting1.txt"), encoding="utf-8") as f:
    test_fold = json.load(f)

train_positions = np.asarray([p for fold in train_folds for p in fold], dtype=np.int64)
test_positions = np.asarray(test_fold, dtype=np.int64)

all_positions = np.concatenate([train_positions, test_positions])
assert len(np.unique(all_positions)) == affinity.size
assert len(np.intersect1d(train_positions, test_positions)) == 0
assert len(train_positions) == 25046 and len(test_positions) == 5010

train_drugs = drug_idx[train_positions]
train_targets = target_idx[train_positions]
test_drugs = drug_idx[test_positions]
test_targets = target_idx[test_positions]

y_train = affinity[train_drugs, train_targets].astype(np.float32)
y_test = affinity[test_drugs, test_targets].astype(np.float32)

print("Training pairs:", len(y_train))
print("Test pairs:", len(y_test))
print("Training fold sizes:", [len(fold) for fold in train_folds])
print("Train/test overlap:", len(np.intersect1d(train_positions, test_positions)))


## 7. Encode SMILES and protein sequences
Each character maps to a positive integer. Zero is reserved for padding. This notebook uses its own Davis-derived vocabularies, and saves them with the model for matching browser inference.


In [ ]:
smiles_chars = sorted(set("".join(drugs)))
protein_chars = sorted(set("".join(targets)))

smiles_vocab = {ch: i + 1 for i, ch in enumerate(smiles_chars)}
protein_vocab = {ch: i + 1 for i, ch in enumerate(protein_chars)}

def encode(sequence, vocab, max_len):
    encoded = np.zeros(max_len, dtype=np.int32)
    for i, ch in enumerate(sequence[:max_len]):
        if ch not in vocab:
            raise ValueError(f"Character {ch!r} is missing from the vocabulary.")
        encoded[i] = vocab[ch]
    return encoded

# Report truncation rather than silently hiding it.
smiles_truncated = sum(len(s) > MAX_SMILES_LENGTH for s in drugs)
protein_truncated = sum(len(s) > MAX_PROTEIN_LENGTH for s in targets)

print("SMILES vocabulary size:", len(smiles_vocab))
print("Protein vocabulary size:", len(protein_vocab))
print("SMILES longer than configured length:", smiles_truncated, "/", len(drugs))
print("Proteins longer than configured length:", protein_truncated, "/", len(targets))
print("Longest SMILES:", max(map(len, drugs)))
print("Longest protein:", max(map(len, targets)))

encoded_drugs = np.stack([encode(s, smiles_vocab, MAX_SMILES_LENGTH) for s in drugs])
encoded_targets = np.stack([encode(s, protein_vocab, MAX_PROTEIN_LENGTH) for s in targets])

X_drug_train = encoded_drugs[train_drugs]
X_target_train = encoded_targets[train_targets]
X_drug_test = encoded_drugs[test_drugs]
X_target_test = encoded_targets[test_targets]

print("Drug train shape:", X_drug_train.shape)
print("Protein train shape:", X_target_train.shape)
print("Drug test shape:", X_drug_test.shape)
print("Protein test shape:", X_target_test.shape)

# Persist preprocessing and mappings in Drive.
np.savez_compressed(
    os.path.join(DATASET_DIR, "davis_browser_preprocessed.npz"),
    encoded_drugs=encoded_drugs,
    encoded_targets=encoded_targets,
    train_positions=train_positions,
    test_positions=test_positions,
    y_train=y_train,
    y_test=y_test
)
with open(os.path.join(MODEL_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump({**CONFIG, "smiles_vocab": smiles_vocab, "protein_vocab": protein_vocab}, f, indent=2)

print("Encoded Davis data and vocabulary config saved to Drive.")


## 8. Build the DeepDTA CNN
This follows the supplied notebook's categorical two-branch architecture: one Embedding + three Conv1D layers for SMILES, one for protein, then concatenation and dense layers.


In [ ]:
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D,
    Concatenate, Dense, Dropout
)
from tensorflow.keras.models import Model

def build_deepdta():
    drug_input = Input(shape=(MAX_SMILES_LENGTH,), dtype="int32", name="drug_input")
    drug_x = Embedding(
        input_dim=len(smiles_vocab) + 1,
        output_dim=EMBEDDING_DIM,
        name="drug_embedding"
    )(drug_input)
    drug_x = Conv1D(NUM_FILTERS, SMILES_KERNEL, activation="relu", name="drug_conv1")(drug_x)
    drug_x = Conv1D(NUM_FILTERS * 2, SMILES_KERNEL, activation="relu", name="drug_conv2")(drug_x)
    drug_x = Conv1D(NUM_FILTERS * 3, SMILES_KERNEL, activation="relu", name="drug_conv3")(drug_x)
    drug_features = GlobalMaxPooling1D(name="drug_global_max_pool")(drug_x)

    protein_input = Input(shape=(MAX_PROTEIN_LENGTH,), dtype="int32", name="protein_input")
    protein_x = Embedding(
        input_dim=len(protein_vocab) + 1,
        output_dim=EMBEDDING_DIM,
        name="protein_embedding"
    )(protein_input)
    protein_x = Conv1D(NUM_FILTERS, PROTEIN_KERNEL, activation="relu", name="protein_conv1")(protein_x)
    protein_x = Conv1D(NUM_FILTERS * 2, PROTEIN_KERNEL, activation="relu", name="protein_conv2")(protein_x)
    protein_x = Conv1D(NUM_FILTERS * 3, PROTEIN_KERNEL, activation="relu", name="protein_conv3")(protein_x)
    protein_features = GlobalMaxPooling1D(name="protein_global_max_pool")(protein_x)

    combined = Concatenate(name="drug_protein_features")([drug_features, protein_features])
    x = Dense(DENSE_UNITS[0], activation="relu", name="dense_1024_a")(combined)
    x = Dropout(DROPOUT_RATE, name="dropout_a")(x)
    x = Dense(DENSE_UNITS[1], activation="relu", name="dense_1024_b")(x)
    x = Dropout(DROPOUT_RATE, name="dropout_b")(x)
    x = Dense(DENSE_UNITS[2], activation="relu", name="dense_512")(x)
    output = Dense(1, activation="linear", name="pkd")(x)

    model = Model(
        inputs=[drug_input, protein_input],
        outputs=output,
        name="DeepDTA_Davis"
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="mse"
    )
    return model

model = build_deepdta()
model.summary()


## 9. Five-fold validation, checkpointing and final training
Each of the five official training folds is used once as validation. We use the median best epoch, then train one final model on all 25,046 development interactions. The held-out test set is not used for model selection.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, BackupAndRestore

cv_best_epochs = []
cv_results = []

for fold_id in range(len(train_folds)):
    print(f"\\n========== CV FOLD {fold_id + 1}/5 ==========")

    val_positions = np.asarray(train_folds[fold_id], dtype=np.int64)
    fit_positions = np.asarray(
        [p for j, fold in enumerate(train_folds) if j != fold_id for p in fold],
        dtype=np.int64
    )

    fit_drugs = drug_idx[fit_positions]
    fit_targets = target_idx[fit_positions]
    val_drugs = drug_idx[val_positions]
    val_targets = target_idx[val_positions]

    Xd_fit = encoded_drugs[fit_drugs]
    Xt_fit = encoded_targets[fit_targets]
    y_fit = affinity[fit_drugs, fit_targets].astype(np.float32)

    Xd_val = encoded_drugs[val_drugs]
    Xt_val = encoded_targets[val_targets]
    y_val = affinity[val_drugs, val_targets].astype(np.float32)

    fold_dir = os.path.join(CHECKPOINT_DIR, f"cv_fold_{fold_id + 1}")
    os.makedirs(fold_dir, exist_ok=True)
    best_path = os.path.join(fold_dir, "best.keras")
    backup_path = os.path.join(fold_dir, "backup")

    cv_model = build_deepdta()
    callbacks = [
        ModelCheckpoint(best_path, monitor="val_loss", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
                      restore_best_weights=True, verbose=1),
        BackupAndRestore(backup_dir=backup_path)
    ]

    hist = cv_model.fit(
        [Xd_fit, Xt_fit], y_fit,
        validation_data=([Xd_val, Xt_val], y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    best_epoch = int(np.argmin(hist.history["val_loss"]) + 1)
    best_val_loss = float(np.min(hist.history["val_loss"]))
    cv_best_epochs.append(best_epoch)
    cv_results.append({
        "fold": fold_id + 1,
        "best_epoch": best_epoch,
        "best_validation_mse": best_val_loss
    })

    with open(os.path.join(fold_dir, "history.json"), "w") as f:
        json.dump(hist.history, f, indent=2)

    print("Best epoch:", best_epoch, "| Best validation MSE:", best_val_loss)
    del cv_model
    tf.keras.backend.clear_session()

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(os.path.join(RESULTS_DIR, "davis_cv_results.csv"), index=False)

FINAL_EPOCHS = max(1, int(np.median(cv_best_epochs)))
print("\\nCV best epochs:", cv_best_epochs)
print("Final training epochs (median):", FINAL_EPOCHS)

# Final refit on every official training interaction.
final_dir = os.path.join(CHECKPOINT_DIR, "final_model")
os.makedirs(final_dir, exist_ok=True)
final_best_path = os.path.join(final_dir, "best.keras")
final_backup_path = os.path.join(final_dir, "backup")

model = build_deepdta()
final_callbacks = [
    ModelCheckpoint(final_best_path, monitor="loss", save_best_only=True, verbose=1),
    BackupAndRestore(backup_dir=final_backup_path)
]

history = model.fit(
    [X_drug_train, X_target_train],
    y_train,
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=final_callbacks,
    verbose=1
)

# Reload the persisted best final model to ensure the saved artifact is the one evaluated.
model = tf.keras.models.load_model(final_best_path)
model.save(os.path.join(MODEL_DIR, "deepdta_davis.keras"))

with open(os.path.join(RESULTS_DIR, "final_training_history.json"), "w") as f:
    json.dump(history.history, f, indent=2)

print("Final model and training history saved in Google Drive.")


## 10. Evaluate once on the untouched Davis test fold
Metrics are calculated on the official held-out test set only after model selection and final training.


In [ ]:
def concordance_index(y_true, y_pred):
    # Pairwise ranking agreement. Tied predictions receive 0.5 credit.
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    concordant = 0.0
    comparable = 0
    for i in range(len(y_true)):
        mask = y_true[i] > y_true
        differences = y_pred[i] - y_pred[mask]
        concordant += np.sum(differences > 0) + 0.5 * np.sum(differences == 0)
        comparable += int(np.sum(mask))
    return float(concordant / comparable) if comparable else float("nan")

def r2m(y_true, y_pred):
    # Defensive implementation: undefined cases return NaN, not a misleading number.
    y_true = np.asarray(y_true, dtype=np.float64).ravel()
    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float("nan")
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    if not np.isfinite(corr):
        return float("nan")
    r2 = corr ** 2
    denom_k = np.sum(y_true ** 2)
    denom_r0 = np.sum((y_pred - np.mean(y_pred)) ** 2)
    if denom_k == 0 or denom_r0 == 0:
        return float("nan")
    k = np.sum(y_pred * y_true) / denom_k
    r0_2 = 1 - np.sum((y_pred - k * y_true) ** 2) / denom_r0
    return float(r2 * (1 - np.sqrt(max(r2 - r0_2, 0.0))))

y_pred = model.predict([X_drug_test, X_target_test], batch_size=BATCH_SIZE, verbose=1).ravel()

ci = concordance_index(y_test, y_pred)
mse = float(mean_squared_error(y_test, y_pred))
r2m_value = r2m(y_test, y_pred)

binary_test = (y_test >= PKD_THRESHOLD).astype(int)
aupr = (
    float(average_precision_score(binary_test, y_pred))
    if len(np.unique(binary_test)) == 2 else float("nan")
)

# Group error analysis is especially important because many Davis targets equal pKd 5.
floor_mask = np.isclose(y_test, 5.0)
above_floor_mask = y_test > 5.0
floor_mse = float(mean_squared_error(y_test[floor_mask], y_pred[floor_mask])) if floor_mask.any() else float("nan")
above_floor_mse = float(mean_squared_error(y_test[above_floor_mask], y_pred[above_floor_mask])) if above_floor_mask.any() else float("nan")

print("========== DAVIS HELD-OUT TEST RESULTS ==========")
print("CI:", ci)
print("MSE:", mse)
print("r2m:", r2m_value)
print("AUPR at pKd >= 7:", aupr)
print("pKd = 5 group count:", int(floor_mask.sum()), "| MSE:", floor_mse)
print("pKd > 5 group count:", int(above_floor_mask.sum()), "| MSE:", above_floor_mse)

metrics = pd.DataFrame([{
    "dataset": "Davis",
    "final_epochs": FINAL_EPOCHS,
    "CI": ci,
    "MSE": mse,
    "r2m": r2m_value,
    "AUPR": aupr,
    "test_pkd5_count": int(floor_mask.sum()),
    "test_pkd5_MSE": floor_mse,
    "test_above5_count": int(above_floor_mask.sum()),
    "test_above5_MSE": above_floor_mse
}])
metrics.to_csv(os.path.join(RESULTS_DIR, "davis_metrics.csv"), index=False)

pd.DataFrame({
    "actual_pKd": y_test,
    "predicted_pKd": y_pred,
    "absolute_error": np.abs(y_test - y_pred)
}).to_csv(os.path.join(RESULTS_DIR, "davis_predictions.csv"), index=False)

print("Metrics and predictions saved in Drive.")


## 11. Save the final model and exact character mappings
The vocabularies and sequence lengths must travel with the model. The browser must use these exact mappings.


In [ ]:
MODEL_FILE = os.path.join(MODEL_DIR, "deepdta_davis.keras")
CONFIG_FILE = os.path.join(MODEL_DIR, "config.json")

model.save(MODEL_FILE)

model_config = {
    **CONFIG,
    "smiles_vocab": smiles_vocab,
    "protein_vocab": protein_vocab,
    "model_file": "deepdta_davis.keras",
    "test_metrics_file": os.path.join(RESULTS_DIR, "davis_metrics.csv")
}

with open(CONFIG_FILE, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=2)

print("Saved model:", MODEL_FILE)
print("Model size (MB):", round(os.path.getsize(MODEL_FILE) / 1e6, 2))
print("Saved config:", CONFIG_FILE)


## 12. Export the trained model for TensorFlow.js
This step is performed only after Python-side test evaluation and model saving. TensorFlow.js conversion compatibility depends on the TensorFlow and converter versions installed in the runtime.


In [ ]:
# Install the converter only at export time, rather than changing the training environment earlier.
!pip -q install tensorflowjs

import shutil
import subprocess

SAVED_MODEL_DIR = os.path.join(MODEL_DIR, "saved_model")
WEB_MODEL_DIR = os.path.join(WEB_DIR, "model")

if os.path.exists(SAVED_MODEL_DIR):
    shutil.rmtree(SAVED_MODEL_DIR)
if os.path.exists(WEB_MODEL_DIR):
    shutil.rmtree(WEB_MODEL_DIR)

os.makedirs(WEB_DIR, exist_ok=True)

# Keras 3 export creates a TensorFlow SavedModel directory.
model.export(SAVED_MODEL_DIR)

conversion = subprocess.run(
    [
        "tensorflowjs_converter",
        "--input_format=tf_saved_model",
        "--output_format=tfjs_graph_model",
        SAVED_MODEL_DIR,
        WEB_MODEL_DIR
    ],
    capture_output=True,
    text=True
)

print("Converter return code:", conversion.returncode)
if conversion.stdout:
    print(conversion.stdout[-3000:])
if conversion.stderr:
    print(conversion.stderr[-5000:])

assert conversion.returncode == 0, "TensorFlow.js conversion failed; preserve the Python .keras model and inspect the converter output."
assert os.path.isfile(os.path.join(WEB_MODEL_DIR, "model.json")), "model.json was not created."

shutil.copy(CONFIG_FILE, WEB_DIR)
print("TensorFlow.js model exported:", WEB_MODEL_DIR)


## 13. Build the browser website
The website loads the TensorFlow.js model and the saved vocabulary config. Unknown characters are rejected rather than silently converted to padding. The interface uses the notebook's pKd threshold of 7.0: below 7 = weaker category; 7 or above = stronger category. This is a research threshold, not a clinical diagnosis.


In [ ]:
WEB_DIR = os.path.join(PROJECT_DIR, "web")
os.makedirs(WEB_DIR, exist_ok=True)

index_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>DeepDTA | Binding Affinity Prediction</title>
    <link rel="stylesheet" href="style.css">
</head>
<body>

    <header class="navbar">
        <div class="logo">
            <div class="logo-icon">🧬</div>
            <div>
                <div class="logo-title">DeepDTA</div>
                <div class="logo-subtitle">Drug–Target Affinity Prediction</div>
            </div>
        </div>
        <div class="nav-badge"><span class="status-dot"></span>AI MODEL ONLINE</div>
    </header>

    <main class="container">

        <section class="hero">
            <div class="hero-badge">Deep Learning for Drug Discovery</div>
            <h1>Predict Drug–Target <span>Binding Affinity</span></h1>
            <p>
                Enter a drug's SMILES representation and a protein sequence
                to estimate their binding affinity using the DeepDTA
                convolutional neural network.
            </p>
        </section>

        <section class="prediction-card">

            <div class="card-header">
                <div>
                    <div class="section-label">PREDICTION</div>
                    <h2>Drug–Target Analysis</h2>
                </div>
                <div class="model-pill">CNN</div>
            </div>

            <div class="input-grid">
                <div class="input-section">
                    <div class="input-title">
                        <div class="input-icon drug-icon">💊</div>
                        <div>
                            <h3>Drug SMILES</h3>
                            <p>Chemical structure representation</p>
                        </div>
                    </div>
                    <textarea id="smiles" placeholder="Example: CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O" spellcheck="false"></textarea>
                    <div class="input-hint">Enter the molecular SMILES notation</div>
                </div>

                <div class="input-section">
                    <div class="input-title">
                        <div class="input-icon protein-icon">🧬</div>
                        <div>
                            <h3>Protein Sequence</h3>
                            <p>Amino acid sequence</p>
                        </div>
                    </div>
                    <textarea id="protein" placeholder="Example: MKTIIALSYIFCLVFADYKDDDDK" spellcheck="false"></textarea>
                    <div class="input-hint">Enter the protein amino acid sequence</div>
                </div>
            </div>

            <button id="predictButton" class="predict-button" onclick="predictAffinity()">
                <span id="buttonText">Predict Binding Affinity</span>
                <span class="button-arrow">→</span>
            </button>

            <div class="result-card">
                <div class="result-left">
                    <div class="result-icon">⚡</div>
                    <div>
                        <div class="result-label">PREDICTED BINDING AFFINITY</div>
                        <div class="result-description">DeepDTA model prediction</div>
                    </div>
                </div>
                <div class="result-value">
                    <span id="resultValue">—</span>
                    <small>pKd</small>
                </div>
            </div>

            <div class="scale-section" id="scaleSection" hidden>
                <div class="category-label" id="categoryLabel"></div>
                <div class="scale-bar">
                    <div class="seg weak"></div>
                    <div class="seg moderate"></div>
                    <div class="seg strong"></div>
                    <div class="marker" id="marker"></div>
                </div>
                <div class="scale-labels">
                    <span>Weak (5 to under 6)</span>
                    <span>Moderate (6 to under 7)</span>
                    <span>Strong (7 and above)</span>
                </div>
            </div>

        </section>

        <section class="info-grid">
            <div class="info-card">
                <div class="info-card-icon">🧪</div>
                <h3>Drug Representation</h3>
                <p>The drug is represented using its SMILES molecular notation and processed as a sequence.</p>
            </div>
            <div class="info-card">
                <div class="info-card-icon">🧬</div>
                <h3>Protein Representation</h3>
                <p>Protein amino acid sequences are encoded into numerical representations for the neural network.</p>
            </div>
            <div class="info-card">
                <div class="info-card-icon">🤖</div>
                <h3>Deep Learning</h3>
                <p>Two convolutional neural network branches learn features from drug and protein sequences.</p>
            </div>
        </section>

    </main>

    <footer>
        <div>
            <strong>DeepDTA</strong>
            <span>Drug–Target Binding Affinity Prediction</span>
        </div>
        <div class="footer-right">Powered by Deep Learning</div>
    </footer>

    <script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.22.0/dist/tf.min.js"></script>
    <script src="script.js"></script>
</body>
</html>
'''

style_css = r'''
* { margin: 0; padding: 0; box-sizing: border-box; }
[hidden] { display: none !important; }

:root {
    --bg: #07111f;
    --card: rgba(17, 31, 52, 0.88);
    --border: rgba(148, 163, 184, 0.14);
    --text: #f8fafc;
    --muted: #94a3b8;
    --primary: #38bdf8;
    --secondary: #8b5cf6;
    --success: #34d399;
}

body {
    font-family: Inter, system-ui, -apple-system, "Segoe UI", sans-serif;
    background: linear-gradient(180deg, #07111f 0%, #091525 45%, #07111f 100%);
    color: var(--text);
    min-height: 100vh;
}

.navbar {
    width: 100%; max-width: 1180px; margin: auto; padding: 26px 24px;
    display: flex; align-items: center; justify-content: space-between;
}
.logo { display: flex; align-items: center; gap: 13px; }
.logo-icon {
    width: 46px; height: 46px; display: flex; align-items: center; justify-content: center;
    border-radius: 14px; font-size: 24px;
    background: linear-gradient(135deg, rgba(56, 189, 248, 0.2), rgba(139, 92, 246, 0.2));
    border: 1px solid rgba(148, 163, 184, 0.18);
}
.logo-title { font-size: 20px; font-weight: 800; }
.logo-subtitle { font-size: 11px; color: var(--muted); margin-top: 2px; }
.nav-badge {
    display: flex; align-items: center; gap: 8px; padding: 8px 13px; border-radius: 999px;
    background: rgba(52, 211, 153, 0.08); border: 1px solid rgba(52, 211, 153, 0.18);
    color: #86efac; font-size: 10px; font-weight: 700; letter-spacing: 0.8px;
}
.status-dot {
    width: 7px; height: 7px; border-radius: 50%; background: var(--success);
    box-shadow: 0 0 10px rgba(52, 211, 153, 0.8);
}

.container { width: min(1100px, calc(100% - 40px)); margin: auto; }

.hero { text-align: center; max-width: 850px; margin: 60px auto 46px; }
.hero-badge {
    display: inline-block; padding: 8px 14px; border-radius: 999px;
    background: rgba(56, 189, 248, 0.07); border: 1px solid rgba(56, 189, 248, 0.18);
    color: #7dd3fc; font-size: 12px; font-weight: 600;
}
.hero h1 { margin-top: 22px; font-size: clamp(38px, 6vw, 60px); line-height: 1.05; font-weight: 850; }
.hero h1 span {
    display: block; background: linear-gradient(90deg, #38bdf8, #8b5cf6);
    -webkit-background-clip: text; background-clip: text; color: transparent;
}
.hero p { max-width: 690px; margin: 22px auto 0; color: var(--muted); font-size: 16px; line-height: 1.75; }

.prediction-card {
    background: var(--card); border: 1px solid var(--border); border-radius: 26px;
    padding: 30px; box-shadow: 0 30px 80px rgba(0, 0, 0, 0.25);
}
.card-header { display: flex; align-items: center; justify-content: space-between; margin-bottom: 28px; }
.section-label { font-size: 10px; font-weight: 800; letter-spacing: 1.8px; color: var(--primary); margin-bottom: 7px; }
.card-header h2 { font-size: 25px; }
.model-pill {
    padding: 8px 14px; border-radius: 10px; background: rgba(139, 92, 246, 0.1);
    border: 1px solid rgba(139, 92, 246, 0.22); color: #c4b5fd; font-size: 11px; font-weight: 800;
}

.input-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
.input-section {
    background: rgba(7, 17, 31, 0.55); border: 1px solid rgba(148, 163, 184, 0.1);
    border-radius: 18px; padding: 20px;
}
.input-title { display: flex; align-items: center; gap: 12px; margin-bottom: 15px; }
.input-icon {
    width: 42px; height: 42px; display: flex; align-items: center; justify-content: center;
    border-radius: 12px; font-size: 20px;
}
.drug-icon { background: rgba(56, 189, 248, 0.1); }
.protein-icon { background: rgba(139, 92, 246, 0.1); }
.input-title h3 { font-size: 15px; font-weight: 700; }
.input-title p { margin-top: 3px; font-size: 11px; color: var(--muted); }
textarea {
    width: 100%; height: 150px; resize: vertical; padding: 16px; border-radius: 13px;
    border: 1px solid rgba(148, 163, 184, 0.12); background: rgba(2, 8, 23, 0.72);
    color: #e2e8f0; outline: none; font-family: Consolas, "Liberation Mono", monospace;
    font-size: 13px; line-height: 1.7;
}
textarea:focus { border-color: rgba(56, 189, 248, 0.45); }
.input-hint { margin-top: 10px; font-size: 10px; color: #64748b; }

.predict-button {
    width: 100%; margin-top: 22px; padding: 16px 20px; border: none; border-radius: 14px;
    cursor: pointer; display: flex; align-items: center; justify-content: center; gap: 12px;
    background: linear-gradient(90deg, #0284c7, #7c3aed); color: white;
    font-size: 14px; font-weight: 750; transition: transform 0.2s ease, opacity 0.2s ease;
}
.predict-button:hover { transform: translateY(-2px); }
.predict-button:disabled { cursor: wait; opacity: 0.65; }
.button-arrow { font-size: 20px; line-height: 0; }

.result-card {
    margin-top: 22px; padding: 20px 22px; border-radius: 17px;
    background: linear-gradient(90deg, rgba(56, 189, 248, 0.07), rgba(139, 92, 246, 0.07));
    border: 1px solid rgba(56, 189, 248, 0.13);
    display: flex; align-items: center; justify-content: space-between;
}
.result-left { display: flex; align-items: center; gap: 13px; }
.result-icon {
    width: 42px; height: 42px; display: flex; align-items: center; justify-content: center;
    border-radius: 12px; background: rgba(52, 211, 153, 0.1); font-size: 18px;
}
.result-label { font-size: 10px; letter-spacing: 1.3px; font-weight: 800; color: #7dd3fc; }
.result-description { margin-top: 4px; color: var(--muted); font-size: 11px; }
.result-value { display: flex; align-items: baseline; gap: 7px; }
.result-value span { font-size: 30px; font-weight: 800; color: #c4b5fd; }
.result-value small { color: var(--muted); font-size: 12px; font-weight: 700; }

.scale-section {
    margin-top: 22px; padding: 22px; border-radius: 17px;
    background: rgba(7, 17, 31, 0.55); border: 1px solid rgba(148, 163, 184, 0.1);
}
.category-label { font-size: 15px; font-weight: 700; margin-bottom: 16px; }
.category-label.weak { color: #fbbf24; }
.category-label.moderate { color: #60a5fa; }
.category-label.strong { color: #34d399; }
.scale-bar { position: relative; display: flex; height: 14px; border-radius: 999px; overflow: visible; }
.seg { height: 100%; }
.seg.weak { flex: 2; background: #f59e0b; border-radius: 999px 0 0 999px; }
.seg.moderate { flex: 1; background: #3b82f6; }
.seg.strong { flex: 4; background: #10b981; border-radius: 0 999px 999px 0; }
.marker {
    position: absolute; top: -6px; left: 0; width: 26px; height: 26px; margin-left: -13px;
    border-radius: 50%; background: white; border: 3px solid #07111f;
    box-shadow: 0 0 0 2px rgba(255, 255, 255, 0.4); transition: left 0.4s ease;
}
.scale-labels { display: flex; justify-content: space-between; margin-top: 12px; font-size: 11px; color: var(--muted); }
.scale-labels span:nth-child(2) { text-align: center; }
.scale-labels span:nth-child(3) { text-align: right; }

.info-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 18px; margin: 65px 0 80px; }
.info-card {
    padding: 24px; border-radius: 18px; background: rgba(17, 31, 52, 0.5);
    border: 1px solid rgba(148, 163, 184, 0.1);
}
.info-card-icon { font-size: 24px; margin-bottom: 15px; }
.info-card h3 { font-size: 15px; }
.info-card p { margin-top: 8px; color: var(--muted); font-size: 12px; line-height: 1.7; }

footer {
    border-top: 1px solid rgba(148, 163, 184, 0.08); width: min(1100px, calc(100% - 40px));
    margin: auto; padding: 25px 0 35px; display: flex; justify-content: space-between;
    color: #64748b; font-size: 11px;
}
footer strong { color: #cbd5e1; margin-right: 7px; }
.footer-right { color: #475569; }

@media (max-width: 850px) {
    .input-grid { grid-template-columns: 1fr; }
    .info-grid { grid-template-columns: 1fr; }
}
@media (max-width: 600px) {
    .nav-badge { display: none; }
    .prediction-card { padding: 18px; }
    .result-card { align-items: flex-start; gap: 15px; flex-direction: column; }
    footer { flex-direction: column; gap: 8px; }
}
'''

script_js = r'''
const SCALE_MIN = 5;
const SCALE_MAX = 11;
const THRESHOLD = 7.0;

let model = null;
let config = null;

async function loadModel() {
    const [configResponse] = await Promise.all([fetch("config.json")]);
    if (!configResponse.ok) throw new Error("Could not load model config.");
    config = await configResponse.json();
    model = await tf.loadGraphModel("model/model.json");
}

const modelReady = loadModel();

function encode(sequence, vocab, maxLength, label) {
    const result = new Array(maxLength).fill(0);
    const used = sequence.slice(0, maxLength);

    for (let i = 0; i < used.length; i++) {
        if (!Object.prototype.hasOwnProperty.call(vocab, used[i])) {
            throw new Error(`${label} contains unsupported character: ${used[i]}`);
        }
        result[i] = vocab[used[i]];
    }
    return result;
}

function getInputName(keyword) {
    const found = model.inputs.find(input =>
        input.name.toLowerCase().includes(keyword)
    );
    if (!found) throw new Error(`Could not identify ${keyword} model input.`);
    return found.name;
}

async function runModel(smiles, protein) {
    const drugName = getInputName("drug");
    const proteinName = getInputName("protein");

    const drugTensor = tf.tensor2d(
        [encode(smiles, config.smiles_vocab, config.max_smiles_length, "SMILES")],
        [1, config.max_smiles_length],
        "int32"
    );
    const proteinTensor = tf.tensor2d(
        [encode(protein, config.protein_vocab, config.max_protein_length, "Protein")],
        [1, config.max_protein_length],
        "int32"
    );

    let output;
    try {
        output = model.predict({
            [drugName]: drugTensor,
            [proteinName]: proteinTensor
        });
        const values = await output.data();
        return values[0];
    } finally {
        drugTensor.dispose();
        proteinTensor.dispose();
        if (output) output.dispose();
    }
}

function showScale(value) {
    const strong = value >= THRESHOLD;
    const label = document.getElementById("categoryLabel");
    label.textContent = strong
        ? "Stronger binding category (pKd ≥ 7)"
        : "Below threshold (pKd < 7)";
    label.className = "category-label " + (strong ? "strong" : "weak");

    const clamped = Math.min(Math.max(value, SCALE_MIN), SCALE_MAX);
    const percent = ((clamped - SCALE_MIN) / (SCALE_MAX - SCALE_MIN)) * 100;
    document.getElementById("marker").style.left = percent + "%";
    document.getElementById("scaleSection").hidden = false;
}

async function predictAffinity() {
    const smiles = document.getElementById("smiles").value.trim();
    const protein = document.getElementById("protein").value.trim().toUpperCase();
    const result = document.getElementById("resultValue");
    const button = document.getElementById("predictButton");
    const buttonText = document.getElementById("buttonText");

    if (!smiles || !protein) {
        alert("Please enter both a Drug SMILES and a Protein Sequence.");
        return;
    }

    result.textContent = "...";
    document.getElementById("scaleSection").hidden = true;
    button.disabled = true;
    buttonText.textContent = "Predicting...";

    try {
        await modelReady;
        const value = await runModel(smiles, protein);
        result.textContent = value.toFixed(4);
        showScale(value);
    } catch (error) {
        console.error(error);
        result.textContent = "Error";
        alert(error.message || "Prediction failed. Check the input and model files.");
    } finally {
        button.disabled = false;
        buttonText.textContent = "Predict Binding Affinity";
    }
}
'''

with open(os.path.join(WEB_DIR, "index.html"), "w") as f:
    f.write(index_html)
with open(os.path.join(WEB_DIR, "style.css"), "w") as f:
    f.write(style_css)
with open(os.path.join(WEB_DIR, "script.js"), "w") as f:
    f.write(script_js)

print("Website files written to", WEB_DIR)


## 14. Python-side sample prediction for browser comparison
We save one held-out test input and its Python prediction. After deployment, use the same input in the website and compare the result within a small numerical tolerance.


In [ ]:
sample_smiles = drugs[test_drugs[0]]
sample_protein = targets[test_targets[0]]
python_prediction = float(
    model.predict([X_drug_test[:1], X_target_test[:1]], verbose=0)[0, 0]
)

sample_file = os.path.join(RESULTS_DIR, "sample_input.json")
sample_record = {
    "smiles": sample_smiles,
    "protein_sequence": sample_protein,
    "python_prediction_pKd": python_prediction,
    "actual_pKd": float(y_test[0])
}
with open(sample_file, "w", encoding="utf-8") as f:
    json.dump(sample_record, f, indent=2)

print("Python prediction:", round(python_prediction, 5))
print("Actual pKd:", round(float(y_test[0]), 5))
print("Sample saved:", sample_file)


## 15. Publish the static website to Hugging Face Spaces
This publishes the browser app and TensorFlow.js model. The public link is created by Hugging Face; it is not a permanent Colab server. Add `HF_TOKEN` in Colab Secrets and enable notebook access before running.


In [ ]:
from huggingface_hub import HfApi

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it in Colab Secrets and enable notebook access.")

# Ensure the published directory has the static Space metadata.
with open(os.path.join(WEB_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(
        "---\n"
        "title: DeepDTA Davis\n"
        "emoji: 🧬\n"
        "colorFrom: blue\n"
        "colorTo: purple\n"
        "sdk: static\n"
        "pinned: false\n"
        "---\n\n"
        "DeepDTA Davis browser application. Predictions run locally in the browser using TensorFlow.js. "
        "Research prototype; predictions are not clinical advice.\n"
    )

api = HfApi(token=HF_TOKEN)
username = api.whoami()["name"]
space_name = "deepdta-davis"
repo_id = f"{username}/{space_name}"

api.create_repo(
    repo_id=repo_id,
    repo_type="space",
    space_sdk="static",
    exist_ok=True
)
api.upload_folder(
    folder_path=WEB_DIR,
    repo_id=repo_id,
    repo_type="space"
)

print("Space repository:", f"https://huggingface.co/spaces/{repo_id}")
print("Public app URL:", f"https://{username}-{space_name}.hf.space")
